# Turkspell ML Stem Classification (Phase 3)
This notebook uses the **Qwen 2.5 7B** language model to automatically classify missing Turkish root words based on their morphotactics (voicing, vowel drops, palatal harmony, etc.).

### Instructions:
1. **Enable GPU:** In the Colab menu, go to `Runtime -> Change runtime type`, select **T4 GPU**, and save.
2. **Upload Data:** Use the folder icon on the left sidebar to upload the `missing_words.txt` file from your local machine to the Colab environment.
3. **Run All:** Go to `Runtime -> Run all` to process the missing words.
4. **Download Result:** The script will generate a `classified_stems.json` file. Download it and move it to your local Turkspell project folder.

In [ ]:
!pip install accelerate bitsandbytes transformers peft trl

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import json
import os

model_id = "Qwen/Qwen2.5-7B-Instruct"

# 4-bit quantization to fit on a free 15GB T4 GPU
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)

print(f"Loading {model_id}...")
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto"
)
print("Model loaded successfully!")

In [ ]:
def classify_stem(word, pos="noun"):
    prompt = f"""Analyze the Turkish {pos} stem '{word}' and output ONLY a valid JSON dictionary with its morphotactic properties.
Determine:
- "voicing": true if the final consonant softens when a vowel is added (e.g., kitap -> kitab-ı), else false.
- "drop": true if the last vowel drops when a suffix is added (e.g., akıl -> akl-ı), else false.
- "inverse_harmony": true if it takes front-vowel suffixes despite back vowels (e.g., saat -> saat-ler), else false.
- "pos": The part of speech (Noun, Verb, Adjective, Adverb, ProperNoun).

Output ONLY the JSON, nothing else."""
    messages = [
        {"role": "system", "content": "You are a Turkish linguistic analyzer. Always output pure JSON without markdown blocks."},
        {"role": "user", "content": prompt}
    ]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors="pt").to(model.device)
    
    with torch.no_grad():
        outputs = model.generate(**inputs, max_new_tokens=100, temperature=0.1, pad_token_id=tokenizer.eos_token_id)
        
    response = tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
    return response.strip()

# Test the classifier
print("Testing classifier on 'akıl':")
print(classify_stem("akıl", "noun"))

In [ ]:
print("Processing missing words...")

if not os.path.exists("missing_words.txt"):
    print("ERROR: Please upload missing_words.txt to the Colab environment first!")
else:
    with open("missing_words.txt", "r", encoding="utf-8") as f:
        words = [line.strip() for line in f if line.strip()]
    
    # We will process the top 500 missing words as a proof of concept to avoid a massive run.
    # You can change the slice below to process more if desired.
    top_words = words[:500]
    
    results = []
    for i, w in enumerate(top_words):
        if i % 50 == 0:
            print(f"Processing word {i}/{len(top_words)}...")
        
        try:
            # Extract the word (ignoring the frequency count if present)
            clean_word = w.split()[0] if ' ' in w else w
            json_out = classify_stem(clean_word)
            # Basic parsing to extract dict
            if '```json' in json_out:
                json_out = json_out.split('```json')[1].split('```')[0]
            data = json.loads(json_out)
            data['lemma'] = clean_word
            results.append(data)
        except Exception as e:
            print(f"Failed to process {w}: {e}")
            
    with open("classified_stems.json", "w", encoding="utf-8") as f:
        json.dump(results, f, ensure_ascii=False, indent=2)
        
    print("Finished! Results saved to classified_stems.json. Please download it.")